<a href="https://colab.research.google.com/github/lhammach/DP-representations/blob/main/dpsgd_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CKA comparison of ResNet18 on CIFAR10 - Baseline vs. Differential Privacy (DP-SGD with Opacus)

In [ ]:
import torch
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

CUDA disponible : True
Nom du GPU : NVIDIA GeForce RTX 2080 Ti


## 0. Setup environment
This section installs dependencies and downloads CIFAR10 dataset.
We ensure reproducibility across baseline and DP training.

## 0.1 Google Colab setup
We download CIFAR10 dataset from FastAI repository and extract it.
This dataset is used for both baseline and DP experiments.

In [2]:
import os
import tarfile
import urllib.request

data_dir = "cifar10"
archive_name = "cifar10.tgz"
url = "https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz"

if not os.path.exists(data_dir):
    print("Téléchargement de CIFAR-10 en cours (160 Mo)...")
    urllib.request.urlretrieve(url, archive_name)
    
    print("Extraction des images en cours...")
    with tarfile.open(archive_name, "r:gz") as tar:
        tar.extractall()
        
    os.remove(archive_name)  # Supprime immédiatement l'archive pour libérer de l'espace
    print("Terminé ! Le dataset est prêt et le dossier est propre.")
else:
    print("Le dossier 'cifar10' existe déjà. Étape ignorée.")

Le dossier 'cifar10' existe déjà. Étape ignorée.


## 0.2 Imports
We import torchvision and Opacus.
Opacus is used only for the differential privacy training pipeline (DP-SGD).

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

from torchvision.datasets import CIFAR10
from torchvision.datasets import ImageFolder
from torchvision import models
import torch.nn as nn
import torch.optim as optim

import opacus
from opacus import PrivacyEngine

import numpy as np
from opacus.utils.batch_memory_manager import BatchMemoryManager

from torchvision.models.resnet import BasicBlock

from tqdm import tqdm 

from pathlib import Path

# Création du répertoire pour sauvegarder les réseaux entraînés
NETWORKS_DIR = Path("./networks")
NETWORKS_DIR.mkdir(parents=True, exist_ok=True)

def get_checkpoint_path(prefix, epsilon, seed, save_dir=NETWORKS_DIR, ext="pth"):
    i = 0
    while True:
        path = save_dir / f"{prefix}_eps{int(epsilon)}_seed{seed}_i{i}.{ext}"
        if not path.exists():
            return path
        i += 1

In [4]:
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("opacus:", opacus.__version__)

torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
opacus: 1.6.0


## 1. Hyperparameters
We define training and DP-related hyperparameters:

- EPOCHS: number of training epochs (shared between baseline and DP)
- LR: learning rate
- EPSILON, DELTA: privacy budget parameters for DP-SGD
- MAX_GRAD_NORM: gradient clipping bound
- BATCH_SIZE: logical batch size
- MAX_PHYSICAL_BATCH_SIZE: memory constraint for DP training

In [5]:
# hyperparamètres
MAX_GRAD_NORM = 1.2
EPSILON = 50.0
DELTA = 1e-8 # << 1 / len(train_dataset)
EPOCHS = 20

LR = 1e-3

# fixer la seed GLOBALEMENT pour la reproductibilité
SEED = 42

import random
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

With DP-SGD, the training requires more memory than with a classical training because we need to store each gradient of each sample (per-sample gradient) before we clip it. We thus use `BatchMemoryManager` which enables to process (MAX_PHYSICAL_BATCH_SIZE =128) examples at a time instead of (BATCH_SIZE = 512) at once.

In [6]:
BATCH_SIZE = 512
MAX_PHYSICAL_BATCH_SIZE = 128

# 2. Dataset : CIFAR10

### Data processing
The CIFAR10 images are initially PIL images. We process the into centered and normalized tensors (3 × 32 × 32).
We use CIFAR10 dataset with standard normalization.
No data augmentation is applied to avoid utility degradation in DP setting.

The same dataset is used for:
- baseline training
- DP-SGD training
- CKA comparison (same inputs required)

In [7]:
# These values, specific to the CIFAR10 dataset, are assumed to be known.
# If necessary, they can be computed with modest privacy budgets.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD_DEV = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD_DEV),
])

# Loading CIFAR10 (avec ImageFolder)
DATA_ROOT = './cifar10'
train_dataset = ImageFolder(root='./cifar10/train', transform=transform)
test_dataset = ImageFolder(root='./cifar10/test', transform=transform)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    drop_last=True # Added to ensure consistent batch sizes
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Dataset prêt !")
print(f"Nombre d'images d'entraînement : {len(train_dataset)}")
print(f"Nombre d'images de test : {len(test_dataset)}")

image, label = train_dataset[0]
print(image.shape)
print(label)

Dataset prêt !
Nombre d'images d'entraînement : 50000
Nombre d'images de test : 10000
torch.Size([3, 32, 32])
0


## 3. Models

We use ResNet18 architecture for both:
- Baseline model (standard SGD)
- DP model (DP-SGD via Opacus)

To ensure compatibility with DP-SGD:
- BatchNorm layers are replaced by GroupNorm
- ReLU inplace operations are disabled
- Residual connections are made non-inplace-safe

In [8]:
# savoir si gpu disponible
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

# utilise le GPU si disponible, sinon le CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA disponible : True
Nom du GPU : NVIDIA GeForce RTX 2080 Ti


### 3.1 Baseline Network
We define a clean ResNet18 model trained without differential privacy.
This model serves as reference for CKA comparison.

In [9]:
baseline_model = models.resnet18(num_classes=10)

# --- Compatibility fix (same architecture as DP model in section 3.2 below) ---

def safe_forward(self, x):
    identity = x

    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    if self.downsample is not None:
        identity = self.downsample(x)

    out = out + identity
    out = self.relu(out)

    return out

BasicBlock.forward = safe_forward

# disable inplace ReLU
for module in baseline_model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False

# send to device
baseline_model = baseline_model.to(device)

# optimizer
baseline_optimizer = optim.RMSprop(baseline_model.parameters(), lr=LR)

### 3.2 DP-SGD Network

In [10]:
model = models.resnet18(num_classes=10)
print(model) # the layers of ResNet18 use BatchNorm2d


# # ------------------ DIAGNOSTIC train ----------------------------------------
# model = nn.Sequential(
#     nn.Flatten(),
#     nn.Linear(3 * 32 * 32, 512),
#     nn.ReLU(),
#     nn.Linear(512, 10),
# )


# BatchNorm2d incompatible avec DP-SGD
from opacus.validators import ModuleValidator
errors = ModuleValidator.validate(model, strict=False)
errors[-1:]



ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

[opacus.validators.errors.ShouldReplaceModuleError("BatchNorm cannot support training with differential privacy. The reason for it is that BatchNorm makes each sample's normalized value depend on its peers in a batch, ie the same sample x will get normalized to a different value depending on who else is on its batch. Privacy-wise, this means that we would have to put a privacy mechanism there too. While it can in principle be done, there are now multiple normalization layers that do not have this issue: LayerNorm, InstanceNorm and their generalization GroupNorm are all privacy-safe since they don't have this property.We offer utilities to automatically replace BatchNorms to GroupNorms and we will release pretrained models to help transition, such as GN-ResNet ie a ResNet using GroupNorm, pretrained on ImageNet")]

#### 3.2.a. Compatibility fixes for DP-SGD

ResNet18 contains operations incompatible with per-sample gradient computation:
- inplace residual additions (`+=`)
- inplace ReLU
- BatchNorm (sample-dependent statistics)

We patch the architecture to make it compatible with Opacus.

In [11]:
# CORRECTION 1 pour train : pas de +=

def safe_forward(self, x):
    identity = x

    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv2(out)
    out = self.bn2(out)

    if self.downsample is not None:
        identity = self.downsample(x)

    out = out + identity   # plus de +=
    out = self.relu(out)

    return out

BasicBlock.forward = safe_forward

In [12]:
# On utilise fix pour remplacer les BatchNorm2d par des GroupNorm
model = ModuleValidator.fix(model) # no exception should be raised

# lors de l'entraînement, GradSampleModule ajoute de hooks pour les per-sample gradients, et il faut avoir ReLU(inplace=False)
for name, module in model.named_modules():
    if isinstance(module, nn.ReLU):
        print(name, module.inplace)
        break

ModuleValidator.validate(model, strict=False)

# check how BatchNorm has been replaced
print(model)

relu True
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): GroupNorm(32, 64, eps=1e-05, affine=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(32, 64, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(32, 64, eps=1e-05, affine=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): GroupNorm(32, 64, eps=1e-05, affine=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): GroupNorm(32, 64, eps=1e-05, affine=Tr

In [13]:
# CORRECTION 2 pour train : désactiver le mode inplace sur TOUTES les ReLU du ResNet
for module in model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False

# vérification
for name, module in model.named_modules():
    if isinstance(module, nn.ReLU):
        print(name, module.inplace)
        break

relu False


In [14]:
model = model.to(device)

## 4. Training
We train the model using two different regimes:
- Standard training (baseline)
- Differentially Private training (DP-SGD with Opacus)

Both use identical architecture and dataset for fair comparison.

In [15]:
def accuracy(preds, labels):
    return (preds == labels).mean()

criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(model.parameters(), lr=LR) # optimizer for GD

### 4.1 Training the Baseline Network

In [16]:
### 4.1 Training the Baseline Network

def train_baseline(model, train_loader, optimizer, epoch, device):
    model.train()

    criterion = nn.CrossEntropyLoss()

    losses = []
    accs = []

    for images, target in train_loader:
        images = images.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        output = model(images)
        loss = criterion(output, target)

        preds = output.argmax(dim=1)
        acc = (preds == target).float().mean().item()

        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        accs.append(acc)

    print(
        f"[Baseline] Epoch {epoch} | "
        f"Loss: {np.mean(losses):.4f} | "
        f"Acc: {np.mean(accs)*100:.2f}%"
    )

for epoch in tqdm(range(EPOCHS), desc="Baseline"):
    train_baseline(baseline_model, train_loader, baseline_optimizer, epoch+1, device)

Baseline:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# # save for cka_analysis.ipynb

baseline_save_path = get_checkpoint_path(
    prefix="baseline_resnet18",
    epsilon=f"{int(EPSILON)}",
    seed=f"{SEED}"
)

torch.save(
    baseline_model.state_dict(),
    baseline_save_path
)

print(f"Baseline sauvegardée dans : {baseline_save_path}")

Mounted at /content/drive


### 4.2 Training DP-SGD

#### 4.2.a Preparation for training
We train the model using two different regimes:
- Standard training (baseline)
- Differentially Private training (DP-SGD with Opacus)

Both use identical architecture and dataset for fair comparison.

`PrivacyEngine` is an object that will follow the DP budget and compute epsilon.

More precisely, `make_private_with_epsilon` computes $\sigma$ = noise_multiplier such that the provided $\epsilon$ is reached at the end of the training.

In [ ]:
privacy_engine = PrivacyEngine(accountant="rdp") # Rényi DP accountant, plus précis que l'accountant de moments pour les budgets élevés

print('------ Before make_private_with_epsilon ------ \n')
print(type(model))
print(type(optimizer))
print(type(train_loader))

model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    epochs=EPOCHS,
    target_epsilon=EPSILON,
    target_delta=DELTA,
    max_grad_norm=MAX_GRAD_NORM,
)

print(f"\n Using sigma={optimizer.noise_multiplier} and C={MAX_GRAD_NORM}")

print('\n ------ After make_private_with_epsilon ------ \n')
print(type(model))
print(type(optimizer))
print(type(train_loader))

/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


------ Before make_private_with_epsilon ------ 

<class 'torchvision.models.resnet.ResNet'>
<class 'torch.optim.rmsprop.RMSprop'>
<class 'torch.utils.data.dataloader.DataLoader'>



 Using sigma=0.372314453125 and C=1.2

 ------ After make_private_with_epsilon ------ 

<class 'opacus.grad_sample.grad_sample_module.GradSampleModule'>
<class 'opacus.optimizers.optimizer.DPOptimizer'>
<class 'opacus.data_loader.DPDataLoader'>


#### 4.2.b Actually training the NN
We track:
- loss
- accuracy
- privacy budget (ε)

Evaluation is done at each epoch.

In [19]:
def train(model, train_loader, optimizer, epoch, device):
    model.train()
    criterion = nn.CrossEntropyLoss()

    losses = []
    top1_acc = []

    with BatchMemoryManager(
        data_loader=train_loader,
        max_physical_batch_size=MAX_PHYSICAL_BATCH_SIZE,
        optimizer=optimizer
    ) as memory_safe_data_loader:

        for i, (images, target) in enumerate(memory_safe_data_loader):
            optimizer.zero_grad()
            images = images.to(device)
            target = target.to(device)

            # compute output
            output = model(images)
            loss = criterion(output, target)

            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = target.detach().cpu().numpy()

            # measure accuracy and record loss
            acc = accuracy(preds, labels)

            losses.append(loss.item())
            top1_acc.append(acc)

            loss.backward()
            optimizer.step()

            # if (i+1) % 200 == 0:
            if i == len(memory_safe_data_loader) - 1:
                # print("i =", i)
                epsilon = privacy_engine.get_epsilon(DELTA)
                print(
                    f"\tTrain Epoch: {epoch} \t"
                    f"Loss: {np.mean(losses):.6f} "
                    f"Acc@1: {np.mean(top1_acc) * 100:.6f} "
                    f"(ε = {epsilon:.2f}, δ = {DELTA})"
                )

def test(model, test_loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    losses = []
    top1_acc = []

    with torch.no_grad():
        for images, target in test_loader:
            images = images.to(device)
            target = target.to(device)

            output = model(images)
            loss = criterion(output, target)
            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = target.detach().cpu().numpy()
            acc = accuracy(preds, labels)

            losses.append(loss.item())
            top1_acc.append(acc)

    top1_avg = np.mean(top1_acc)

    print(
        f"\tTest set:"
        f"Loss: {np.mean(losses):.6f} "
        f"Acc: {top1_avg * 100:.6f} "
    )
    return np.mean(top1_acc)

In [20]:
for epoch in tqdm(range(EPOCHS), desc="Epoch", unit="epoch"):
    train(model, train_loader, optimizer, epoch + 1, device)

Epoch:   0%|          | 0/20 [00:00<?, ?epoch/s]

sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


	Train Epoch: 1 	Loss: 2.443228 Acc@1: 20.553380 (ε = 14.49, δ = 1e-05)
	Train Epoch: 2 	Loss: 1.759033 Acc@1: 39.695100 (ε = 18.23, δ = 1e-05)
	Train Epoch: 3 	Loss: 1.715350 Acc@1: 46.296123 (ε = 21.12, δ = 1e-05)
	Train Epoch: 4 	Loss: 1.690515 Acc@1: 50.161918 (ε = 23.60, δ = 1e-05)
	Train Epoch: 5 	Loss: 1.679377 Acc@1: 53.333540 (ε = 25.82, δ = 1e-05)
	Train Epoch: 6 	Loss: 1.690818 Acc@1: 55.118501 (ε = 27.91, δ = 1e-05)
	Train Epoch: 7 	Loss: 1.658904 Acc@1: 56.660133 (ε = 29.86, δ = 1e-05)
	Train Epoch: 8 	Loss: 1.653074 Acc@1: 57.471736 (ε = 31.68, δ = 1e-05)
	Train Epoch: 9 	Loss: 1.639800 Acc@1: 58.527096 (ε = 33.48, δ = 1e-05)
	Train Epoch: 10 	Loss: 1.632201 Acc@1: 59.404746 (ε = 35.15, δ = 1e-05)
	Train Epoch: 11 	Loss: 1.629824 Acc@1: 60.181542 (ε = 36.77, δ = 1e-05)
	Train Epoch: 12 	Loss: 1.619421 Acc@1: 60.412816 (ε = 38.38, δ = 1e-05)
	Train Epoch: 13 	Loss: 1.622015 Acc@1: 61.055045 (ε = 39.91, δ = 1e-05)
	Train Epoch: 14 	Loss: 1.595658 Acc@1: 61.649613 (ε = 41.40

In [ ]:
# # save for cka_analysis.ipynb

dp_save_path = get_checkpoint_path(
    prefix="dp_resnet18",
    epsilon=f"{int(EPSILON)}",
    seed=f"{SEED}"
)

torch.save(
    model._module.state_dict(),
    dp_save_path
)

print(f"Modèle DP sauvegardé dans : {dp_save_path}")

In [30]:
!find . -name "*.pth"

./dp_resnet18_eps50.pth
./checkpoints/dp_resnet18_eps50.pth
./checkpoints/baseline_resnet18.pth
./baseline_resnet18.pth
